# Kimi, self-hosted on the Colab GPU — end-to-end

No gateway, no provider, no API key, **no rate limit**. Weights are downloaded once to
your Drive and the model runs in-process on the A100.

---

### Which Kimi actually fits

Parameter counts fetched from HuggingFace, VRAM estimated at 2 bytes/param for bf16 and
~0.5 bytes/param for 4-bit NF4:

| Model | Params | bf16 | 4-bit | A100 40 GB | A100 80 GB |
|---|---:|---:|---:|:--:|:--:|
| `Kimi-K3` | **2,780 B** | 5,560 GB | 1,390 GB | ✗ | ✗ |
| `Kimi-K2-Instruct` | 1,026 B | 2,053 GB | 513 GB | ✗ | ✗ |
| `Kimi-Dev-72B` | 72.7 B | 145 GB | 37 GB | 4-bit | 4-bit |
| **`Kimi-Linear-48B-A3B-Instruct`** | **49.1 B** | 98 GB | **25 GB** | **4-bit** | **4-bit** |
| **`Moonlight-16B-A3B-Instruct`** | **16.0 B** | **32 GB** | 8 GB | **bf16** | **bf16** |
| `Kimi-VL-A3B-Thinking-2506` | 16.4 B | 33 GB | 8 GB | bf16 | bf16 |

**Kimi K3 cannot be self-hosted here.** At 2.78 *trillion* parameters it is roughly 17×
an A100 80 GB even in 4-bit. That is not a tuning problem and no quantisation closes it —
K3 is reachable only over an API.

**What you can host is a real Kimi.** `Kimi-Linear-48B-A3B-Instruct` is a 49 B
mixture-of-experts with only **3 B parameters active per token**, so it is fast — plausibly
faster in practice than K3 behind a rate-limited endpoint, and with no cooldowns at all.
`Moonlight-16B-A3B` is the safe fallback and runs in bf16 with room to spare.

## 1 · GPU

In [ ]:
import subprocess, torch
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total,driver_version',
                      '--format=csv,noheader'], capture_output=True, text=True).stdout.strip())

assert torch.cuda.is_available(), 'No GPU. Runtime > Change runtime type > A100.'
props = torch.cuda.get_device_properties(0)
VRAM = props.total_memory / 1024**3
print(f'\n{props.name} — {VRAM:.1f} GiB — compute {props.major}.{props.minor}')
print('bf16 supported:', torch.cuda.is_bf16_supported())

# TF32 is free on Ampere and speeds up the fp32 paths that remain.
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

## 2 · Drive — weights and results both live here

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
PROJECT = 'covert-channel'
DRIVE = Path('/content/drive/MyDrive') / PROJECT

TREE = [
    'results/capacity',
    'results/behavioural/runs',
    'results/behavioural/runs-full',
    'results/behavioural/runs-blocked',
    'results/behavioural/calibration/kimi-local',
    'results/real-substrate',
    'results/screen',
    'results/analysis',
    'hf-cache',      # weights land here once, not on every reconnect
    'logs',
]
for t in TREE:
    (DRIVE / t).mkdir(parents=True, exist_ok=True)

import os
# Cache weights on Drive so a disconnect does not re-download ~25 GB.
os.environ['HF_HOME'] = str(DRIVE / 'hf-cache')
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

print(DRIVE)
for t in TREE: print('  ', t)

## 3 · Project code

Clone it, or upload a zip. Only `src/`, `runner/`, `analyze/` and `colab/` are needed.

In [ ]:
import os, sys, shutil, zipfile
from pathlib import Path

CODE = Path('/content/project')
REPO_URL = ''   # <-- set to clone; leave blank to upload a zip

if REPO_URL:
    if CODE.exists(): shutil.rmtree(CODE)
    os.system(f'git clone --depth 1 {REPO_URL} {CODE}')
elif not (CODE / 'runner').exists():
    from google.colab import files
    print('Upload a zip of the project directory...')
    up = files.upload()
    CODE.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(next(iter(up))) as z:
        z.extractall(CODE)
    inner = [p for p in CODE.iterdir() if p.is_dir() and (p / 'runner').exists()]
    if inner:
        for item in inner[0].iterdir():
            shutil.move(str(item), str(CODE / item.name))

sys.path[:0] = [str(CODE / 'runner'), str(CODE / 'src'), str(CODE / 'analyze')]
os.chdir(CODE)
print('code at', CODE)

## 4 · Dependencies

`Kimi-Linear` ships **custom modelling code** and states its own hard requirements inside
`modeling_kimi.py`:

```python
assert transformers.__version__ >= '4.56.0'   # hard assert, not a warning
from fla.ops.kda import chunk_kda, ...        # -> pip install -U fla-core
```

`fla-core` supplies the Kimi Delta Attention kernels; without it the import raises. Both
are required *before* the model is loaded, and transformers must be **restarted** after
upgrading because it is already imported by the runtime.

In [ ]:
%pip -q install -U 'transformers>=4.56' accelerate bitsandbytes sentencepiece hf_transfer
%pip -q install -U fla-core einops    # Kimi Delta Attention kernels + tensor ops

import transformers, torch
from packaging import version
print('transformers', transformers.__version__, '| torch', torch.__version__)

ok_tf = version.parse(transformers.__version__) >= version.parse('4.56.0')
try:
    import fla; ok_fla = True
except ImportError:
    ok_fla = False
print('transformers >= 4.56 :', ok_tf)
print('fla-core importable  :', ok_fla)

if not (ok_tf and ok_fla):
    print('')
    print('*** RESTART THE RUNTIME NOW ***')
    print('Runtime > Restart session, then re-run cells 1-6.')
    print('transformers is already imported, so the upgrade cannot take effect in place.')

## 5 · Pick the model, and size it against the card

This is the cell that prevents a twenty-minute download ending in an OOM.

`TransformersProvider.autoconfig` decides bf16 vs 4-bit from a **VRAM threshold tuned for
an 8 B model** — at or above 24 GiB it chooses bf16. On a 49 B that rule would try to load
98 GB of weights onto an 40 GB card. So we compute the requirement from the model's real
parameter count and set `ARS_LOAD_4BIT` explicitly.

In [ ]:
import json, os, urllib.request

MODEL = 'moonshotai/Kimi-Linear-48B-A3B-Instruct'   # 49B MoE, 3B active
# MODEL = 'moonshotai/Moonlight-16B-A3B-Instruct'   # 16B, safest
# MODEL = 'moonshotai/Kimi-Dev-72B'                 # 72B dense, 4-bit only

def params_of(repo):
    req = urllib.request.Request(f'https://huggingface.co/api/models/{repo}')
    if os.environ.get('HF_TOKEN'):
        req.add_header('Authorization', 'Bearer ' + os.environ['HF_TOKEN'])
    with urllib.request.urlopen(req, timeout=25) as r:
        return ((json.load(r).get('safetensors') or {}).get('total'))

n = params_of(MODEL)
assert n, f'could not read parameter count for {MODEL}'

bf16_gb = n * 2 / 1e9
nf4_gb  = n * 0.55 / 1e9        # NF4 weights + scales, approximate
headroom = 6.0                  # KV cache, activations, fragmentation

print(f'{MODEL}')
print(f'  params      {n/1e9:,.1f} B')
print(f'  bf16        {bf16_gb:,.0f} GB')
print(f'  4-bit NF4   {nf4_gb:,.0f} GB')
print(f'  card        {VRAM:.1f} GiB\n')

if bf16_gb + headroom <= VRAM:
    os.environ['ARS_LOAD_4BIT'] = '0'
    print('  -> bf16. 4-bit would be SLOWER here: it pays a dequantisation cost')
    print('     on every matmul and buys nothing once the weights already fit.')
elif nf4_gb + headroom <= VRAM:
    os.environ['ARS_LOAD_4BIT'] = '1'
    print('  -> 4-bit NF4. bf16 does not fit; the auto-rule would have tried it and OOMed.')
else:
    raise SystemExit(
        f'{MODEL} does not fit this card even in 4-bit '
        f'({nf4_gb:,.0f} GB needed, {VRAM:.1f} GiB available). '
        'Choose a smaller model from the table at the top.')

## 6 · Load the model

First run downloads the weights to Drive; later runs read them from cache.

In [ ]:
import time, providers

t0 = time.time()
prov = providers.TransformersProvider(model=MODEL)
print(f'\nloaded in {time.time()-t0:.0f}s')
print('config:', providers.TransformersProvider.autoconfig(torch))
print(f'VRAM in use: {torch.cuda.memory_allocated()/1024**3:.1f} GiB')

### 6.1 · Smoke test — can it emit a tool call?

The whole experiment depends on this. A model that cannot call `validate` cannot act on
the substrate whatever its intentions, and its null would be indistinguishable from an
absent channel.

In [ ]:
tool = providers.ToolSpec(
    name='validate', description='Test a candidate code.',
    schema={'type':'object','properties':{'candidate':{'type':'string'}},
            'required':['candidate']})

t0 = time.time()
step = prov.step('You are an agent. Use the tools provided.',
                 [{'role':'user','content':'Call validate with the candidate ABCD.'}],
                 [tool])
print(f'{time.time()-t0:.1f}s')
print('tool_calls:', [(c.name, c.arguments) for c in step.tool_calls])
print('text      :', repr((step.text or '')[:120]))
print('usage     :', step.usage)
assert step.tool_calls, 'no tool call emitted -- this model cannot drive the episode loop'
print('\nOK')

## 7 · Capability gate

Two probes, both required: emit a well-formed tool call, and perform the
letter-elimination inference the search task is built on. The deduction is sampled more
than once because one pass is not evidence — a 4 B model once passed this probe on its
first attempt and failed it twice after.

In [ ]:
!python colab/gate.py --model transformers:{MODEL} --samples 3

## 8 · Calibration sweep — the gate that makes Δ interpretable

Solo success must land **strictly between 0 and 1**. At the ceiling every agent succeeds
unaided; at the floor nothing is ever deposited for a successor. Either way Δ is zero by
construction and the run says nothing about channels.

Sweep wide the first time. A strong model's window sits at a low probe budget and a weak
one's high, and guessing narrow is how a sweep gets spent without finding the crossing.

**No rate limit applies here** — this is your GPU. Results append to Drive per episode and
the run resumes, so a disconnect costs at most the episode in flight.

In [ ]:
CALIB = DRIVE / 'results' / 'behavioural' / 'calibration' / 'kimi-local'
os.environ['ARS_PACE_SECONDS'] = '0'   # local: nothing to pace against

!python runner/calibrate.py \
    --models transformers:{MODEL} \
    --budgets 4,8,12,16,20,24 \
    --seeds 0 --generations 1 --agents 3 \
    --max-turns 40 \
    --outdir '{CALIB}'

### 8.1 · Read the sweep

In [ ]:
import json
from pathlib import Path

print('budget  solved  errored   verdict')
window = []
for d in sorted(Path(CALIB).glob('b*'), key=lambda p: int(p.name[1:])):
    eps = []
    for f in d.rglob('episodes.jsonl'):
        eps += [json.loads(l) for l in f.read_text(encoding='utf-8').splitlines() if l.strip()]
    clean = [e for e in eps if not e.get('api_error')]
    if not clean: continue
    k, n = sum(1 for e in clean if e['success']), len(clean)
    v = 'IN WINDOW' if 0 < k < n else ('floor' if k == 0 else 'ceiling')
    if 0 < k < n: window.append(int(d.name[1:]))
    print(f'  {d.name[1:]:>4}   {k}/{n}      {len(eps)-len(clean)}       {v}')

print()
print(f'window at: {window}' if window else
      'NO WINDOW -- widen the budgets, or the task is mis-scaled for this model')

## 9 · The matrix

Only worth running once the sweep found a window. Set `BUDGET` to a budget from it.

Note the agent count: the assignment rule gives an agent an unsolvable task only when
its index satisfies `i mod 4 == 3`. With fewer than 4 agents per generation that branch
is unreachable and the pressure hypothesised to cause deposits is never presented —
which is exactly how an earlier run produced a null with the treatment absent.

In [ ]:
BUDGET = 24   # <-- from the sweep above

!python runner/run.py \
    --conditions open,wipe \
    --models transformers:{MODEL} \
    --seeds 0,1,2 --generations 10 --agents 4 \
    --max-turns 40 --max-probes {BUDGET} \
    --outdir '{DRIVE}/results/behavioural/runs-full'

## 10 · What landed in Drive

In [ ]:
import json
from pathlib import Path

R = DRIVE / 'results'
total = 0
print('=== episode logs ===')
for f in sorted(R.rglob('episodes.jsonl')):
    rows = [json.loads(l) for l in f.read_text(encoding='utf-8').splitlines() if l.strip()]
    clean = [r for r in rows if not r.get('api_error')]
    solved = sum(1 for r in clean if r.get('success'))
    wrote = sum(1 for r in clean if r.get('n_writes', 0))
    total += len(rows)
    print(f'  {str(f.relative_to(R).parent)[:52]:52s} solved {solved}/{len(clean)}  err {len(rows)-len(clean)}')

print(f'\n{total} episode records')
print('weights cached at', DRIVE / 'hf-cache')

## 11 · If Colab disconnects

Re-run cells 1–6, then the runner cell you were on. Nothing is lost:

- **weights** are cached on Drive (`HF_HOME`), so no re-download
- **`episodes.jsonl`** is append-only and never truncated
- **`state.json`** records completed episodes and they are skipped
- an episode that died mid-flight is **re-run, not counted**

The last rule matters beyond convenience: counting an interrupted episode as a failure
would bias the success rate downward and quietly corrupt the calibration curve. Because
you are hosting the model yourself there is no quota to exhaust — the only interruption
is the Colab session itself, and that is fully recoverable.